# Bankroll Simulation

Monte Carlo bankroll paths under different Kelly fractions, with drawdown analysis
and ruin probability estimation. Demonstrates why half-Kelly is the practical optimum
and shows the catastrophic risk of overbetting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from speculator_ev_engine.core.bankroll import (
    simulate_bankroll, estimate_ruin_probability,
    analyze_drawdowns, kelly_fraction_sweep,
)
from speculator_ev_engine.core.kelly import binary_kelly

## 1. Bankroll Paths Under Different Fractions

In [ ]:
# 5% edge on even-money bets
p = 0.525
odds = 1.0
full_k = binary_kelly(p, odds).fraction

fig, ax = plt.subplots(figsize=(12, 6))
for frac_mult, label in [(0.25, 'Quarter'), (0.5, 'Half'), (1.0, 'Full'), (2.0, 'Double')]:
    f = full_k * frac_mult
    paths = simulate_bankroll(p, odds, f, n_steps=2000, n_simulations=50, seed=42)
    for i in range(min(5, paths.shape[0])):
        ax.plot(paths[i], alpha=0.3, linewidth=0.8)
    # Plot median path
    median = np.median(paths, axis=0)
    ax.plot(median, linewidth=2, label=f'{label} Kelly (f={f:.3f})')

ax.set_yscale('log')
ax.set_xlabel('Steps')
ax.set_ylabel('Bankroll (log scale)')
ax.set_title('Bankroll Paths Under Different Kelly Fractions (p=0.525, b=1.0)')
ax.legend()
plt.show()

## 2. Ruin Probability by Kelly Fraction

In [ ]:
fractions = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.15, 0.20])
ruin_probs = []
for f in fractions:
    result = estimate_ruin_probability(p, odds, f, n_steps=5000, n_simulations=2000, seed=42)
    ruin_probs.append(result.ruin_probability)

plt.figure(figsize=(10, 6))
plt.plot(fractions, ruin_probs, 'ro-')
plt.axvline(full_k, color='blue', linestyle='--', label=f'Full Kelly = {full_k:.3f}')
plt.axvline(full_k / 2, color='green', linestyle='--', label=f'Half Kelly = {full_k/2:.3f}')
plt.xlabel('Kelly Fraction')
plt.ylabel('Ruin Probability')
plt.title('Ruin Probability vs Kelly Fraction (p=0.525, b=1.0)')
plt.legend()
plt.show()

## 3. Drawdown Analysis

In [ ]:
for frac_mult, label in [(0.5, 'Half'), (1.0, 'Full'), (1.5, '1.5x')]:
    f = full_k * frac_mult
    paths = simulate_bankroll(p, odds, f, n_steps=5000, n_simulations=1000, seed=42)
    dd = analyze_drawdowns(paths)
    print(f'{label} Kelly (f={f:.3f}):')
    print(f'  Max drawdown: {dd.max_drawdown:.2%}')
    print(f'  Mean drawdown: {dd.mean_drawdown:.2%}')
    print(f'  Max duration: {dd.max_drawdown_duration} steps')
    print()

## 4. Optimal Fraction via Empirical Sweep

In [ ]:
sweep_results = kelly_fraction_sweep(p, odds, n_steps=5000, n_simulations=1000, seed=42)
fracs = [r[0] for r in sweep_results]
growths = [r[1] for r in sweep_results]
ruins = [r[2] for r in sweep_results]

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(fracs, growths, 'b-', label='Mean Log Growth')
ax1.set_xlabel('Kelly Fraction')
ax1.set_ylabel('Mean Log Growth', color='b')

ax2 = ax1.twinx()
ax2.plot(fracs, ruins, 'r-', label='Ruin Probability')
ax2.set_ylabel('Ruin Probability', color='r')
ax2.set_ylim(0, 1)

# Find peak growth
best_idx = np.argmax(growths)
ax1.axvline(fracs[best_idx], color='green', linestyle='--', 
            label=f'Peak growth at f={fracs[best_idx]:.2f}')

plt.title('Empirical Kelly Sweep: Growth vs Ruin')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()
print(f'Theoretical Kelly fraction: {full_k:.4f}')
print(f'Empirical peak growth at: {fracs[best_idx]:.4f}')